# Chronos-2 Rolling Forecast on Nifty

Same pipeline as the Sensex notebook, run on Nifty this time.
This lets us check later whether our findings hold on a second index,
not just Sensex.

In [1]:
import pandas as pd
import numpy as np
import torch
from chronos import BaseChronosPipeline

In [2]:
df = pd.read_csv("../data/nifty_clean.csv", parse_dates=["Date"])
df.tail()

,Date,Open,High,Low,Close
4560,2026-04-23,24202.349609,24310.199219,24134.800781,24173.050781
4561,2026-04-24,24100.550781,24206.000000,23813.650391,23897.949219
4562,2026-04-27,23945.449219,24130.699219,23936.199219,24092.699219
4563,2026-04-28,24049.900391,24181.800781,23957.050781,23995.699219
4564,2026-04-29,24096.900391,24334.699219,24059.949219,24177.650391


In [3]:
prices = df["Close"].values
prices.shape

(4565,)

In [4]:
pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu",
    torch_dtype=torch.float32,
)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

In [5]:
context_len = 512
horizon = 30
quantile_levels = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

step = 10
start = context_len
end = len(prices) - horizon

all_forecasts = []
all_actuals = []
all_dates = []

for i in range(start, end, step):
    context = prices[i - context_len:i]
    actual = prices[i:i + horizon]

    context_tensor = torch.tensor(context, dtype=torch.float32)

    quantiles, mean = pipeline.predict_quantiles(
        inputs=[context_tensor],
        prediction_length=horizon,
        quantile_levels=quantile_levels,
    )

    q_values = quantiles[0][0].numpy()

    all_forecasts.append(q_values)
    all_actuals.append(actual)
    all_dates.append(df["Date"].iloc[i])

    if i % 100 == 0:
        print("done up to window", i, "of", end)

In [6]:
all_forecasts = np.array(all_forecasts)
all_actuals = np.array(all_actuals)

all_forecasts.shape, all_actuals.shape

((403, 30, 11), (403, 30))

In [7]:
np.save("nifty_forecasts.npy", all_forecasts)
np.save("nifty_actuals.npy", all_actuals)

# also save the dates so we can line up regimes later
np.save("nifty_window_dates.npy", np.array(all_dates))

print("saved")

saved
